# [LAB] 과정 1 코드 다시 읽기

과정 1에서 만든 코드 네 가지를 다시 실행하고, AI에게 구조를 설명받으며 기초 구성요소를 되짚습니다. 새로 배우는 코드는 없습니다. `# ✍️ CHANGE_HERE:` 줄을 바꿔 다시 실행하고, 장 끝의 "AI에게 물어보기"로 코드가 어떻게 돌아가는지 설명받습니다.

- **2장 LangChain으로 호출하기**: 메시지 목록을 모델에 보내고 답을 한 번에, 또 조각으로 받습니다.
- **3장 도구 호출**: 모델이 도구를 고르고, 코드가 실행하고, 결과를 다시 모델에 넣는 반복을 봅니다.
- **4장 Researcher와 Writer**: 조사하는 Agent와 글 쓰는 Agent를 LangGraph 그래프로 잇습니다.
- **5장 RAG 체인**: 약관 PDF에서 관련 내용을 찾아 답변에 더합니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다. 모델을 호출하는 셀은 결과가 나오기까지 수 초에서 수십 초가 걸리고, 답은 실행할 때마다 조금씩 다릅니다.

In [ ]:
%cd ../../../course/Chapter02/notebooks

**목차**

- 1. 실습 방법
- 2. LangChain으로 호출하기
  - 2.1 역할 메시지로 한 번에 받기
  - 2.2 조각으로 받기
  - 2.3 AI에게 물어보기
- 3. 도구 호출
  - 3.1 도구 정의
  - 3.2 모델에 도구 연결하기
  - 3.3 도구 호출 루프
  - 3.4 질문해 보기
  - 3.5 AI에게 물어보기
- 4. Researcher와 Writer
  - 4.1 검색 도구
  - 4.2 공유 상태
  - 4.3 Researcher 노드
  - 4.4 도구를 쓸지 정하는 조건
  - 4.5 Writer 노드
  - 4.6 그래프 연결
  - 4.7 실행하기
  - 4.8 AI에게 물어보기
- 5. RAG 체인
  - 5.1 벡터 저장소와 retriever 만들기
  - 5.2 프롬프트와 RAG 체인 연결하기
  - 5.3 질문하고 답변 받기
  - 5.4 AI에게 물어보기
- 6. 과제: 약관 검색 도구 Agent
- 7. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env(["EMBED_PROVIDER"])

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고친 뒤 바로 위 환경 점검 셀을 다시 실행합니다. 무엇을 고칠지 모르겠으면 그 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

LangChain 디버그 모드를 켭니다. 체인과 그래프가 실행될 때 단계마다 입력과 출력이 셀 아래에 출력됩니다.

In [ ]:
from langchain_core.globals import set_debug

# 출력이 너무 길면 True를 False로 바꾸고 이 셀을 다시 실행합니다
set_debug(True)

<details><summary>강사용 메모</summary>

- 원본: `course/Chapter02/code/raw/00-review/` 네 파일. TODO가 없어 모두 예제 + AI에게 물어보기로 만들었습니다. 네 파일은 M1 코드와 같습니다(00 = M1 01-llm-api/03_langchain, 01 = M1 05-multi-agent/01_researcher_writer 정답, 02 = M1 02-langchain/02_tool_calling 정답, 03 = M1 03-rag/04_langchain_rag 정답).
- 장 순서는 원본 번호(00→01→02→03)가 아니라 00→02→01→03입니다. 도구 호출 루프를 먼저 봐야 LangGraph의 `ToolNode`·조건부 엣지가 읽힙니다(목차 제안대로 결정).
- 커리큘럼 ③ 개발 환경 구성(VS Code, CodeMate, venv, .env)은 맨 위 환경 점검 셀과 강사 진행으로 처리합니다. 3일차에 PC를 다시 셋팅해야 하는지는 미확인입니다.
- 커리큘럼 ④ "server, client 기초 이해" 실습은 이 노트북에 없습니다. 3-2·3-3이 맡습니다(2026-09-28 사용자 결정).
- CodeMate가 노트북에서 선택한 셀을 읽는지는 강의장에서 확인하지 않았습니다. 못 읽으면 셀 코드를 지시 아래에 붙여 넣게 합니다.
- 사외 실측(2026-09-28, `COURSE_ENV=out`): 새 커널로 전체 실행 약 1분(4장 글 작성이 가장 김). 사내 게이트웨이 소요 시간과 tool calling·임베딩 동작은 미확인입니다.

</details>

## 1. 실습 방법

코드 셀에서 손댈 곳은 주석으로 표시되어 있습니다.

- `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.

실행한 뒤에는 **성공 기준**과 비교해 봅니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다.

**막혔을 때 AI에게 도움받는 프롬프트 힌트** 오류가 난 셀의 코드와 오류 메시지 전체를 붙여 넣고, 원인과 고칠 곳을 쉬운 말로 알려 달라고 합니다.

## 2. LangChain으로 호출하기

`get_llm()`으로 준비한 모델에 메시지 목록을 보냅니다. `SystemMessage`에는 모델의 역할을, `HumanMessage`에는 질문을 넣습니다. 같은 메시지를 `invoke()`로 한 번에 받고, `stream()`으로 조각씩 받아 봅니다.

슬라이드의 체인(`prompt | model | parser`)과 모양은 다르지만, 같은 모델을 메시지 목록으로 호출하는 코드입니다.

준비 코드입니다. 메시지 종류(`SystemMessage`, `HumanMessage`)를 불러오고 `llm = get_llm()`으로 모델을 준비합니다. 펼치지 않고 실행만 해도 됩니다.

In [ ]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage

from config.llm import get_llm

load_dotenv()


llm = get_llm()

<details><summary>강사용 메모</summary>

- 원본 `00_langchain.py`(raw). raw와 솔루션이 같습니다. 슬라이드 p4 코드(`ChatPromptTemplate` 체인)와 모양이 다릅니다.

</details>

### 2.1 역할 메시지로 한 번에 받기

**✍️ CHANGE_HERE: 설명받고 싶은 개념으로 바꿔 봅니다**

먼저 그대로 두 셀을 실행해 답을 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 "LLM"을 과정 1에서 배운 다른 개념(예: RAG, Agent)으로 바꾸고, 이 셀과 다음 셀을 다시 실행합니다. 따옴표는 지우지 않습니다.

In [ ]:
# ── 기본 invoke ───────────────────────────────────────────────────────────────
# SystemMessage: LLM의 역할/행동 지침 설정
# HumanMessage: 사용자 입력
messages = [
    SystemMessage(content="너는 초보자를 위한 AI 튜터야."),
    # ✍️ CHANGE_HERE: 설명받고 싶은 개념으로 바꿔 봅니다
    HumanMessage(content="LLM이 무엇인지 3문장으로 설명해줘."),
]

response = llm.invoke(messages)
print(response.content)
print()

**성공 기준**

- 🟢 바꾼 개념을 설명하는 답변 텍스트가 한 번에 출력되었다

### 2.2 조각으로 받기

In [ ]:
# ── Streaming ─────────────────────────────────────────────────────────────────
# stream()은 토큰 단위로 응답을 yield함
# end='' 로 줄바꿈 없이 실시간 출력
for chunk in llm.stream(messages):
    print(chunk.content, end="", flush=True)
print()

**성공 기준**

- 🟢 같은 주제의 답변이 한 번 더, 조금씩 이어서 출력되었다
- ⛔ 두 번째 답변이 몇 글자마다 줄이 바뀐다 → `print` 안의 `end=""`가 지워졌는지 확인합니다

**볼 지점**
- 두 셀 모두 같은 `messages`를 보냅니다. 첫 셀은 답을 다 만든 뒤 한 번에, 둘째 셀은 만들어지는 대로 조각씩 받습니다.
- 답이 두 번 나오는 까닭은 모델을 두 번 호출했기 때문입니다. 두 답의 문장은 조금 다를 수 있습니다.

### 2.3 AI에게 물어보기

**프롬프트 힌트** 두 셀을 붙여 넣고, `invoke`와 `stream`이 같은 `messages`를 받는데 출력이 어떻게 다른지 설명해 달라고 합니다. 코드는 고치지 말라고 적습니다.

**성공 기준**

- 🟢 `invoke`는 완성된 답 하나를, `stream`은 조각(chunk)을 차례로 돌려준다고 짚었다
- 🟢 `end=""`가 조각을 줄바꿈 없이 이어 붙인다고 설명했다
- ⛔ 코드를 고치거나 새로 써 왔다 → 고치지 말라고 다시 적습니다

## 3. 도구 호출

모델에게 날씨 조회 도구와 계산 도구를 알려 주고, 질문에 맞는 도구를 모델이 고르게 합니다. 모델은 도구를 직접 실행하지 않고 "어떤 도구를 어떤 값으로" 호출할지만 돌려줍니다. 실행은 코드가 하고, 결과를 다시 모델에 넣어 최종 답을 받습니다.

`@tool`을 붙인 함수의 docstring(함수 첫 줄의 설명 글)은 모델에게 주는 도구 설명이 되고, 타입 힌트(`city: str`)는 인자 형식이 됩니다. 날씨 도구는 실제로 조회하지 않고 서울·부산·제주 세 도시의 정해 둔 날씨만 돌려줍니다.

**왜 배우나:** 도구를 고르고, 실행하고, 결과를 다시 넣는 이 반복이 Agent의 기본 동작입니다. 과정 2의 MCP도 이 반복에 도구를 연결하는 방식입니다.

준비 코드입니다. 메시지 종류와 도구를 만드는 `@tool`을 불러옵니다. 접혀 있으니 실행만 합니다.

In [ ]:
"""
Tool Calling: LLM이 외부 도구를 자율적으로 선택·실행하기

핵심 흐름:
  사용자 질문 → LLM이 필요한 도구 결정 → 도구 실행 → 결과 반환

"""

import os

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import tool

from config.llm import get_llm

load_dotenv();

<details><summary>강사용 메모</summary>

- 원본 `02_tool_calling.py`(raw). 노트북에서만 손봤습니다(patch): ① docstring의 `실행:` 줄 삭제 ② `05-langgraph/03_tool_node.py 참고` 경로 주석 삭제 ③ raw에만 있는 `print(response)`(응답 객체 전체라 한 번에 수십 줄)를 빼고 `for call in …` 안에 `print(call)`을 넣음. M1 1-3과 같은 모양으로, 호출 정보(name·args·id)만 짧게 보입니다(목차 5절 질문 3, 가독성 기준으로 결정). 솔루션에는 두 줄 모두 없습니다.
- 사내 게이트웨이에서 tool calling(`bind_tools`) 동작은 M1 1-3과 같은 조건입니다. 미확인이면 성공 기준과 출력 설명으로 진행합니다.
- `calculate`는 `eval`을 씁니다(builtins 비움). 수업용 가짜 도구입니다.
- 사외 실측(2026-09-28): 복합 질문에서 `get_weather`가 서울·부산 두 번 호출됐고, CHANGE_HERE 예시 "제주 기온에 2를 곱해줘"는 2번 모두 `get_weather` → `calculate('20 * 2')` → 40°C 순서였습니다.

</details>

### 3.1 도구 정의

In [ ]:
# ── 도구 정의 (@tool 데코레이터) ────────────────────────────────────────────────
# 함수의 docstring이 LLM에게 전달되는 도구 설명이 된다.
# 타입 힌트가 JSON Schema로 변환되어 파라미터 형식을 LLM에게 알려준다.


@tool
def get_weather(city: str) -> str:
    """지정한 도시의 현재 날씨를 조회한다."""
    fake_data = {
        "서울": "맑음, 기온 22°C",
        "부산": "흐림, 기온 24°C",
        "제주": "비, 기온 20°C",
    }
    return fake_data.get(city, f"{city}의 날씨 데이터를 찾을 수 없습니다.")


@tool
def calculate(expression: str) -> str:
    """수학 수식을 계산한다. 예: '2 + 3 * 4'"""
    try:
        result = eval(expression, {"__builtins__": {}})  # noqa: S307
        return str(result)
    except Exception as e:
        return f"계산 오류: {e}"


tools = [get_weather, calculate]

### 3.2 모델에 도구 연결하기

In [ ]:
# ── LLM에 도구 바인딩 ─────────────────────────────────────────────────────────
# bind_tools()는 LLM 호출 시마다 도구 스키마를 자동으로 포함시킨다.
llm = get_llm()
llm_with_tools = llm.bind_tools(tools)

# 도구 이름 → 실행 함수 매핑
tool_map = {t.name: t for t in tools}

### 3.3 도구 호출 루프

In [ ]:
# ── 수동 도구 호출 루프 ───────────────────────────────────────────────────────
# LLM 응답에 tool_calls가 있으면 실행 후 결과를 다시 LLM에 전달한다.
# LangGraph의 ToolNode가 이 루프를 자동화한다.


def run_with_tools(question: str) -> str:
    messages = [HumanMessage(content=question)]

    while True:
        response = llm_with_tools.invoke(messages)
        messages.append(response)
        if not response.tool_calls:
            return response.content

        for call in response.tool_calls:
            print(call)
            tool_fn = tool_map[call["name"]]
            result = tool_fn.invoke(call["args"])
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=call["id"],
                )
            )

### 3.4 질문해 보기

**✍️ CHANGE_HERE: 두 도구를 이어 쓰는 질문으로 바꿔 봅니다**

처음에는 고치지 않고 실행합니다. 질문 세 개를 차례로 보내고, 질문마다 모델을 두 번 이상 호출하므로 조금 기다립니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 질문을 날씨와 계산이 함께 필요한 질문(예: "제주 기온에 2를 곱해줘")으로 바꿔 이 셀을 다시 실행합니다. 바꿀 것은 따옴표 안 글뿐입니다.

In [ ]:
# ── 실행 예시 ─────────────────────────────────────────────────────────────────

print("=== 날씨 질문 ===")
print(run_with_tools("서울 날씨 알려줘"))

print("\n=== 수학 계산 ===")
print(run_with_tools("(123 + 456) * 2를 계산해줘"))

print("\n=== 복합 질문 ===")
# ✍️ CHANGE_HERE: 두 도구를 이어 쓰는 질문으로 바꿔 봅니다
print(run_with_tools("서울과 부산 날씨를 비교해줘"))

# LangGraph ToolNode 방식으로 전환하면 이 루프가 필요 없어진다.

**성공 기준**

- 🟢 질문마다 도구 호출 정보 줄(`{'name': …, 'args': …}`)이 먼저 출력되고, 그 뒤에 최종 답이 출력된다
- 🟢 날씨 질문에는 `get_weather`, 계산 질문에는 `calculate`가 호출되었다
- 🟢 복합 질문에서는 `get_weather` 호출 정보 줄이 도시 수만큼 나온다
- 🟢 바꾼 질문에서는 `get_weather` 다음에 `calculate` 호출 정보 줄이 나오고, 최종 답에 계산 결과가 들어 있다
- ⛔ 답에 "날씨 데이터를 찾을 수 없습니다"만 나온다 → 질문의 도시가 서울·부산·제주 중 하나인지 확인합니다

**볼 지점**
- 호출 정보 줄의 `name`은 모델이 고른 도구, `args`는 모델이 채운 인자, `id`는 호출 번호입니다.
- 도구를 실행한 것은 `run_with_tools()` 안의 코드입니다. 실행 결과는 `ToolMessage`로 대화에 들어가고, 모델이 그것을 읽고 다음 판단을 합니다.
- 날씨와 계산을 이어 쓰는 질문에서는 모델이 날씨 결과를 받은 뒤에야 계산할 값을 알 수 있습니다. 그래서 반복이 한 번 더 돕니다.

<details><summary>강사용 메모</summary>

- LangGraph의 `ToolNode`가 이 루프를 자동화합니다(다음 장 `01_researcher_writer`의 `researcher_tools`).

</details>

### 3.5 AI에게 물어보기

**프롬프트 힌트** "도구 호출 루프" 셀을 붙여 넣고, `while` 반복이 언제 끝나는지와 `ToolMessage`에 `tool_call_id`를 넣는 이유를 설명해 달라고 합니다. 수정 없이 근거가 되는 줄만 짚어 달라고 적습니다.

**성공 기준**

- 🟢 모델 응답에 `tool_calls`가 없을 때 `return`으로 반복이 끝난다고 짚었다
- 🟢 `tool_call_id`는 도구 결과가 어느 호출에 대한 답인지 모델이 짝지을 수 있게 한다고 설명했다
- ⛔ 고친 코드를 내놓았다 → 설명만 필요하다고 다시 적습니다
- ⛔ "도구를 한 번 호출하면 끝난다"고 답했다 → 복합 질문의 출력을 붙여 넣고 반복이 몇 번 돌았는지 다시 묻습니다

## 4. Researcher와 Writer

조사하는 Agent(Researcher)와 글 쓰는 Agent(Writer)를 LangGraph 그래프 하나로 잇습니다. Researcher는 검색 도구 `search_web`을 쓸 수 있고, Writer는 도구 없이 조사 결과만 받아 글을 씁니다. 두 Agent는 같은 상태(State)를 봅니다. 조사 결과는 `research_result`, 완성된 글은 `draft`에 담깁니다.

앞 장에서 직접 짠 도구 호출 반복을 여기서는 그래프가 맡습니다. 도구 실행은 `ToolNode`가, "도구를 더 쓸지, Writer로 넘길지"는 조건 함수가 정합니다. 검색 도구는 실제로 검색하지 않고 정해 둔 글을 돌려줍니다.

그래프는 맨 아래 "실행하기" 셀에서 처음 실행됩니다. 위 셀들을 차례로 실행한 뒤 마지막 셀에서 결과를 봅니다.

준비 코드입니다. LangGraph 도구를 불러오고 `llm = get_llm()`으로 모델을 준비합니다. 내용은 앞 장들과 비슷하니 실행하고 넘어갑니다.

In [ ]:
from typing import Annotated

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()

<details><summary>강사용 메모</summary>

- 원본 `01_researcher_writer.py`. raw와 솔루션은 코드가 같고, raw에만 이전 강사 메모 주석 13줄이 있어 솔루션을 썼습니다(가독성 우선). 메모 요지: 도구 함수에는 LLM이 이해할 수 있는 설명을 자연어로 쓴다 / `bind_tools`는 모델 출력이 도구 호출인지 직접 답인지의 양식을 정한다.
- 쓰지 않는 import `tools_condition`이 접힌 준비 코드에 남아 있습니다.
- 반복 횟수 제한이 코드에 없습니다. 그래서 ⛔에 "실행 중지(■)"를 적었습니다.
- 사외 실측(2026-09-28, `COURSE_ENV=out`): 약 3,300자 글이 나왔고, 가짜 검색 결과의 "주요 플레이어" 목록(OpenAI, Google DeepMind)은 글에 들어가지 않았습니다. 그래서 성공 기준에 검색 결과 문구를 넣지 않았습니다.
- 원본은 마지막 답만 `print(...content)`로 출력합니다. 노트북은 결과 변수를 그래프가 드러나는 이름(`researcher_writer_result`)으로 바꾸고, 메시지 목록을 `results`로 꺼내 `pretty_print()`로 출력합니다(2026-10-17 규칙, dev/AGENTS.md).

</details>

### 4.1 검색 도구

In [ ]:
# ── 도구 ──────────────────────────────────────────────────────────────────────


@tool
def search_web(query: str) -> str:
    """웹에서 주제를 조사한다."""
    # 실제 구현에서는 Tavily, SerpAPI 등을 사용한다.
    return (
        f'["{query}" 검색 결과]\n'
        f"- 해당 주제의 주요 동향: 최근 급격한 성장세를 보이고 있음\n"
        f"- 핵심 기술 요소: 트랜스포머 아키텍처, 대규모 데이터 학습\n"
        f"- 주요 플레이어: OpenAI, Anthropic, Google DeepMind\n"
        f"- 미래 전망: 향후 3년간 지속적인 발전이 예상됨"
    )

### 4.2 공유 상태

In [ ]:
# ── 상태 ──────────────────────────────────────────────────────────────────────


class State(TypedDict):
    messages: Annotated[list, add_messages]
    research_result: str  # Researcher가 수집한 정보
    draft: str  # Writer가 작성한 초안

### 4.3 Researcher 노드

In [ ]:
# ── Researcher Agent ──────────────────────────────────────────────────────────
# 역할: 주제를 조사해 research_result에 저장한다.

researcher_llm = llm.bind_tools([search_web])
researcher_tool_node = ToolNode([search_web])


def researcher_node(state: State) -> dict:
    system = SystemMessage(
        content=(
            "너는 전문 리서처야. "
            "search_web 도구를 사용해 주어진 주제를 철저히 조사하고, "
            "수집한 정보를 정리해서 제공해줘."
        )
    )
    response = researcher_llm.invoke([system] + state["messages"])
    return {"messages": [response]}


def researcher_tools_node(state: State) -> dict:
    # ToolNode 실행 후 research_result에 도구 결과를 저장
    tool_result = researcher_tool_node.invoke(state)
    last_tool_msg = tool_result["messages"][-1]
    return {
        "messages": tool_result["messages"],
        "research_result": last_tool_msg.content,
    }

### 4.4 도구를 쓸지 정하는 조건

In [ ]:
def researcher_condition(state: State):
    """Researcher가 도구 호출을 완료하면 writer로 넘어간다."""
    last = state["messages"][-1]
    if hasattr(last, "tool_calls") and last.tool_calls:
        return "researcher_tools"
    return "writer"

### 4.5 Writer 노드

In [ ]:
# ── Writer Agent ──────────────────────────────────────────────────────────────
# 역할: Researcher의 조사 결과를 바탕으로 완성된 글을 작성한다.


def writer_node(state: State) -> dict:
    system = SystemMessage(
        content=(
            "너는 전문 작가야. "
            "제공된 리서치 결과를 바탕으로 독자가 이해하기 쉬운 글을 작성해줘. "
            "서론, 본론, 결론 구조를 갖춰줘."
        )
    )
    context = f"[리서치 결과]\n{state.get('research_result', '')}"
    response = llm.invoke(
        [
            system,
            HumanMessage(content=context),
            *state["messages"],
        ]
    )
    return {"messages": [response], "draft": response.content}

### 4.6 그래프 연결

In [ ]:
# ── 그래프 조립 ───────────────────────────────────────────────────────────────

builder = StateGraph(State)
builder.add_node("researcher", researcher_node)
builder.add_node("researcher_tools", researcher_tools_node)
builder.add_node("writer", writer_node)

builder.add_edge(START, "researcher")
builder.add_conditional_edges("researcher", researcher_condition)
builder.add_edge("researcher_tools", "researcher")
builder.add_edge("writer", END)

app = builder.compile()

### 4.7 실행하기

**✍️ CHANGE_HERE: 조사할 주제를 바꿔 봅니다**

먼저 그대로 실행해 결과를 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 요청을 다른 주제(예: "반도체 패키징 기술의 현황과 전망을 조사해서 글로 써줘.")로 바꿔 이 셀을 다시 실행합니다.

In [ ]:
researcher_writer_result = app.invoke(
    {
        "messages": [
            # ✍️ CHANGE_HERE: 조사할 주제를 바꿔 봅니다
            HumanMessage(content="AI Agent 기술의 현황과 전망을 조사해서 글로 써줘.")
        ],
        "research_result": "",
        "draft": "",
    }
)
results = researcher_writer_result["messages"]
for m in results:
    m.pretty_print()

**성공 기준**

- 🟢 마지막 `Ai Message` 칸에 서론·본론·결론 구조의 글이 출력되었다
- 🟢 주제를 바꾸면 글의 주제도 바뀐다
- ⛔ 셀이 몇 분이 지나도 끝나지 않는다 → 실행을 중지(■)하고 요청 문장을 다시 봅니다

**볼 지점**
- 흐름은 researcher → researcher_tools → researcher → writer 순서입니다. Researcher가 도구 결과를 보고 한 번 더 판단한 뒤, 조건 함수가 writer로 보냅니다.
- 검색 도구는 어떤 주제든 정해 둔 같은 모양의 결과를 돌려줍니다. Writer는 이 조사 결과를 받아 쓰지만, 모델이 알고 있는 내용도 더하므로 글이 조사 결과보다 훨씬 깁니다.

### 4.8 AI에게 물어보기

**프롬프트 힌트** "도구를 쓸지 정하는 조건" 셀과 "그래프 연결" 셀을 붙여 넣고, 노드가 실행되는 순서를 화살표로 그려 설명해 달라고 합니다. 설명만 받고 코드는 그대로 둔다고 적습니다.

**성공 기준**

- 🟢 researcher → researcher_tools → researcher → writer → END 순서를 그렸다
- 🟢 마지막 메시지에 `tool_calls`가 없으면 writer로 간다는 점을 짚었다
- ⛔ 새 코드를 써 왔다 → 코드는 그대로 두고 순서만 설명해 달라고 다시 적습니다
- ⛔ researcher_tools에서 writer로 바로 간다고 그렸다 → `add_edge("researcher_tools", "researcher")` 줄을 다시 보라고 묻습니다

## 5. RAG 체인

약관 PDF를 청크로 나눠 벡터 저장소에 넣고, 질문과 가까운 청크 3개를 찾아 답변에 더하는 RAG 체인을 실행합니다. 검색 → 프롬프트 → 모델 → 출력 파서를 `|`로 잇고, 답변은 조각씩 받습니다.

PDF를 읽고 청크로 나누는 일은 접힌 준비 코드에서 합니다. 임베딩 요청이 여러 번 가므로 벡터 저장소를 만드는 셀은 조금 걸립니다.

준비 코드입니다. 필요한 도구를 불러오고, 약관 PDF를 읽어 청크(`chunks`)로 나눕니다. 이 셀은 모델을 호출하지 않습니다. 이 셀에서 `is not a valid file or url` 오류가 나면 이 노트북 옆 `data` 폴더에 약관 PDF가 있는지 확인합니다.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
import warnings

# 도구 교체 안내 경고를 숨깁니다(실습과 관계없는 안내입니다).
warnings.filterwarnings("ignore", message=".*langchain-community.*")
from langchain_community.document_loaders import PDFMinerLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

from config.embeddings import get_embeddings
from config.llm import get_llm

load_dotenv()


PDF_PATH = Path("data/meritz_pet_insurance.pdf")
loader = PDFMinerLoader(str(PDF_PATH.resolve()))
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
chunks = text_splitter.split_documents(documents)

<details><summary>강사용 메모</summary>

- 원본 `03_langchain_rag.py`(raw). raw와 솔루션이 같습니다. 노트북에서만 손봤습니다(M1 1-4와 같은 처리): ① `PDF_PATH`를 `data/…` 상대 경로로 ② `langchain-community` 종료 예고 경고 숨김 ③ `Chroma().delete_collection()` 추가(한 커널에서 다시 실행하면 청크가 쌓여 검색 결과가 중복됨) ④ `while True` + `input()` 반복을 질문 변수 하나로 한 번만 실행.
- 사내 임베딩(`EMBED_PROVIDER=samsung`)으로는 실행하지 않았습니다(M1 1-4와 같은 조건).

</details>

### 5.1 벡터 저장소와 retriever 만들기

In [ ]:
# 이 셀을 다시 실행해도 같은 청크가 겹쳐 쌓이지 않게, 앞서 만든 벡터 저장소를 비웁니다.
Chroma().delete_collection()

embeddings = get_embeddings()
vectorstore = Chroma.from_documents(
    chunks,
    embedding=embeddings,
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

### 5.2 프롬프트와 RAG 체인 연결하기

In [ ]:
llm = get_llm()
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 문서 기반 QA 어시스턴트입니다. 주어진 context만 근거로 답하세요.",
        ),
        ("human", "context:\n{context}\n\nquestion:\n{question}"),
    ]
)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

### 5.3 질문하고 답변 받기

**✍️ CHANGE_HERE: 약관에 묻고 싶은 질문으로 바꿔 봅니다**

먼저 그대로 실행해 답변을 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 질문을 약관에 있을 법한 다른 질문(예: "보험금을 청구하려면 무엇이 필요해?")으로 바꿔 이 셀만 다시 실행합니다. 따옴표는 지우지 않습니다.

In [ ]:
print("RAG ready")
# ✍️ CHANGE_HERE: 약관에 묻고 싶은 질문으로 바꿔 봅니다
question = "펫보험 보장 내용이 뭐야?"
for message in rag_chain.stream(question):
    print(message, end="")
print("\n")

**성공 기준**

- 🟢 `RAG ready` 아래에 답변이 한 번에 나오지 않고 조금씩 이어서 출력된다
- 🟢 답변이 약관 내용(보장 범위, 청구 서류 등)을 근거로 한다
- ⛔ 약관과 상관없는 일반론만 나온다 → 질문이 약관에 있을 법한 내용인지 확인합니다
- ⛔ `NameError`(예: `name 'rag_chain' is not defined`)가 난다 → 5장 준비 코드와 5.1·5.2 셀을 먼저 실행합니다

**볼 지점**
- 질문 하나가 두 갈래로 갑니다. 한쪽은 retriever로 가서 약관 청크 3개를 찾고(`context`), 다른 쪽은 그대로 `question`이 됩니다.
- system 지시문이 "주어진 context만 근거로 답하세요"라고 요구합니다. 답변이 약관 문장에 기대고 있는지 봅니다.

### 5.4 AI에게 물어보기

**프롬프트 힌트** "프롬프트와 RAG 체인 연결하기" 셀을 붙여 넣고, 질문 하나가 `rag_chain` 안에서 어디로 흘러가 답변이 되는지 차례대로 설명해 달라고 합니다. 수정은 필요 없다고 적습니다.

**성공 기준**

- 🟢 질문이 retriever와 `RunnablePassthrough()` 양쪽으로 가고, 검색 결과가 `format_docs`로 합쳐져 `context`가 된다고 짚었다
- 🟢 `context`와 `question`이 프롬프트를 채우고 → 모델 → `StrOutputParser` 순서로 간다고 설명했다
- ⛔ 체인을 고쳐 왔다 → 수정 없이 흐름만 설명해 달라고 다시 적습니다

## 6. 과제: 약관 검색 도구 Agent

이 장의 학습 목표를 코드로 구현해 봅니다. 과제 파일과 자세한 안내(할 일, 프롬프트 힌트, 실행 방법)는 노트북 밖 폴더에 있습니다.

**학습 목표**

- `config.check`로 설정을 점검하고 ⛔ 항목을 고쳐 모델 호출까지 통과시킬 수 있다
- `get_embeddings()`와 Chroma로 검색기를 만들고 `@tool search_policy`로 감쌀 수 있다
- `tools_condition`과 `add_edge("tools", "agent")`로 도구를 반복해 호출하는 그래프를 만들 수 있다
- 약관 질문에서만 도구를 호출하고 인사에는 호출하지 않는 것을 `pretty_print`로 보여 줄 수 있다

**터미널 과제**

1. VS Code 탐색기에서 `course/Chapter02/tasks/3-1/README.md`를 열고 안내대로 AI와 함께 작성합니다.
2. VS Code 메뉴 **터미널 → 새 터미널**을 열고(저장소 맨 위 폴더에서 열립니다) 아래 명령으로 실행합니다.

```
uv run --env-file .env python course/Chapter02/tasks/3-1/task.py
```

3. 다 됐으면 AI에게 **"3-1 과제 채점해 줘"**라고 말합니다. 학습 목표마다 🟢 통과 / ⛔ 수정 필요 / ❔ 확인 불가와 다음 할 일이 나옵니다.

## 7. 정리

과정 1의 기초 구성요소 네 가지를 다시 실행하고 구조를 설명받았습니다.

| 장 | 구성요소 | 다시 확인한 것 |
|---|---|---|
| 2. LangChain으로 호출하기 | 모델 호출 | 메시지 목록을 `invoke`로 한 번에, `stream`으로 조각씩 받음 |
| 3. 도구 호출 | Tool Calling | 모델이 도구를 고르고, 코드가 실행하고, 결과를 다시 넣는 반복 |
| 4. Researcher와 Writer | LangGraph | 도구 실행은 `ToolNode`가, 다음 차례는 조건 함수가 정함 |
| 5. RAG 체인 | RAG | 질문으로 청크를 찾아 `context`로 넣고 답변을 받음 |

과정 2의 MCP, A2A, LangGraph 심화는 모두 이 네 가지 위에 올라갑니다.

값을 바꿔 다시 실행하고, 결과를 성공 기준(🟢/⛔)과 비교해 확인했습니다.